<a href="https://colab.research.google.com/github/BhavyeNijhawan/swarkavach/blob/main/notebooks/02_antispoof_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. Anti-spoofing on real data

Trains `RawNetLite` on ASVspoof 2019 LA, fits the LFCC-GMM and GFCC-GMM
baselines beside it, measures EER and minimum detection cost on the official
eval partition, then runs all three on In-the-Wild to see how far the numbers
move when the data is not a benchmark.

**Needs:** a GPU. Runtime -> Change runtime type -> T4 GPU. Without one the
notebook still finishes: it drops to a small subset, fewer epochs and CPU
training, and marks the result as reduced so nothing is reported as a full run.

**Time:** 35 to 60 minutes on a T4 with the default caps. Roughly 2.5 hours on
CPU with the reduced settings.

**Writes back:**

- `data/results/antispoof_results.json`, the EER and min t-DCF grid plus DET
  curve points, in the shape `evaluate.py` writes and the console reads
- `data/results/antispoof_branch_scores.json`, one P(synthetic) per generated
  corpus call, which notebook 04 folds into the fusion vector
- `data/models/antispoof_rawnet.pt`, the checkpoint, about 400 kB

`.gitignore` excludes `data/models/*.pt`. If you want the checkpoint in the
repository, `git add -f data/models/antispoof_rawnet.pt`. It is small enough
that this is reasonable, and it is what makes the laptop able to run the
real-data anti-spoof branch without a GPU.

## Two things worth stating before any code runs

**Why 8 kHz.** Every model in this project runs at telephone bandwidth,
because the system is aimed at phone calls and phone calls are narrowband.
ASVspoof ships 16 kHz flac, so the loader resamples to 8 kHz. This throws away
the 4 to 8 kHz band, which is exactly where a lot of vocoder artefacts live,
and the EER gets worse for it. That is the correct trade: a detector tuned on
detail no telephone network transmits reports a number it cannot reproduce in
deployment. ASVspoof 2021 made the same point the expensive way, when models
trained on studio audio fell apart the moment a real codec touched the signal.

**Why speaker-disjoint.** ASVspoof 2019 LA already splits train, dev and eval
by speaker, and the eval partition also uses different attack algorithms
(A07 to A19) from train and dev (A01 to A06). Both matter. A shared speaker
lets the network memorise a voice instead of learning what synthesis sounds
like; a shared attack lets it memorise one vocoder. The verification cell
below checks the speaker part rather than trusting the documentation.

In [ ]:
# ---------------------------------------------------------------------------
# Get the repository and install the Colab stack.
# Re-running this cell is safe: if the clone is already on disk it pulls.
# ---------------------------------------------------------------------------

# CHANGE THIS to your own fork before you run anything.
REPO_URL = "https://github.com/BhavyeNijhawan/svarkavach.git"
REPO_DIR = "svk_repo"   # NOT "swarkavach": a directory on sys.path whose name matches the package shadows the real install
BRANCH = "main"

import os
import sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO = BASE / REPO_DIR

if (REPO / "pyproject.toml").exists():
    print(f"{REPO} is already cloned, pulling instead")
    !git -C "{REPO}" pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} "{REPO}"

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("cwd:", os.getcwd())

!pip install -q -r requirements-colab.txt
!pip install -q -e .

import swarkavach
print("swarkavach", swarkavach.__version__, "ready")

## Runtime budget

Everything below scales off `HAS_GPU`. The caps are there so a free Colab
session finishes rather than being reclaimed halfway through epoch 7.

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import torch

from swarkavach import config

config.ensure_dirs()          # data/results and data/models must exist before we plot

HAS_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")
GPU = torch.cuda.get_device_name(0) if HAS_GPU else "none"

if HAS_GPU:
    EPOCHS = 12
    BATCH = 64
    MAX_TRAIN = 25380        # the whole LA train partition
    MAX_EVAL = 71237         # the whole LA eval partition
    MAX_ITW = 31779          # all of In-the-Wild
    GMM_UTTS = 3000          # utterances per class for fitting the GMMs
    NUM_WORKERS = 2
else:
    EPOCHS = 4
    BATCH = 16
    MAX_TRAIN = 4000
    MAX_EVAL = 6000
    MAX_ITW = 3000
    GMM_UTTS = 600
    NUM_WORKERS = 0
    print("NO GPU. Running a reduced configuration:")
    print(f"  epochs {EPOCHS}, batch {BATCH}, train cap {MAX_TRAIN}, eval cap {MAX_EVAL}")
    print("  every exported number is tagged reduced=True so the report cannot")
    print("  quote it as a full run.")

REDUCED = not HAS_GPU
print(f"\ndevice {DEVICE}  gpu {GPU}  torch {torch.__version__}")
print(f"free disk {shutil.disk_usage('.').free / 1e9:.1f} GB")

## Find the data

Notebook 01 wrote `dataset_manifest.json` with the paths. If it is not there
(fresh session, or you skipped 01) the cell searches the usual places, and if
it finds nothing it falls back to the generated corpus and says so.

In [ ]:
import json
from pathlib import Path

MANIFEST = Path("data/results/dataset_manifest.json")
manifest = {}
if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    print(f"manifest from notebook 01: SUBSET={manifest.get('subset')}")


def find_la_root():
    """Directory containing ASVspoof2019_LA_cm_protocols."""
    hinted = (manifest.get("asvspoof") or {}).get("root")
    candidates = [Path(hinted)] if hinted else []
    for base in (Path("/content/raw/asvspoof2019_la"), Path("data/raw/asvspoof2019_la"),
                 Path("/content/drive/MyDrive/swarkavach"), Path("/content")):
        if base.exists():
            candidates.extend(p.parent for p in base.glob("**/ASVspoof2019_LA_cm_protocols"))
    for c in candidates:
        if c and (Path(c) / "ASVspoof2019_LA_cm_protocols").is_dir():
            return Path(c)
    return None


def find_itw_root():
    hinted = ((manifest.get("in_the_wild") or {}).get("status") or {}).get("root")
    for base in ([Path(hinted)] if hinted else []) + [
            Path("/content/raw/in_the_wild"), Path("data/raw/in_the_wild")]:
        if base and Path(base).exists() and any(Path(base).glob("**/*.wav")):
            return Path(base)
    return None


LA_ROOT = find_la_root()
ITW_ROOT = find_itw_root()
print("ASVspoof LA root:", LA_ROOT)
print("In-the-Wild root:", ITW_ROOT)

if LA_ROOT is None:
    print("\nNo ASVspoof on disk. Run notebook 01 first, or set SUBSET='none' "
          "there on purpose. This notebook will train on the generated corpus "
          "instead and label every number as generated audio.")

## Protocol parsing

A CM protocol line is five fields: speaker, utterance id, a dash, the attack
id, and the key.

```
LA_0079 LA_T_1138215 - - bonafide
LA_0079 LA_T_1271820 - A01 spoof
```

The attack id is worth keeping rather than collapsing to a binary label,
because a single aggregate EER hides the fact that a detector can be near
perfect on six of the thirteen eval attacks and useless on A17. The per-attack
breakdown goes into the export.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

PROTOCOLS = {
    "train": "ASVspoof2019.LA.cm.train.trn.txt",
    "dev": "ASVspoof2019.LA.cm.dev.trl.txt",
    "eval": "ASVspoof2019.LA.cm.eval.trl.txt",
}


@dataclass
class Item:
    path: str
    label: int          # 1 = spoof, 0 = bonafide, matching evaluate.eer
    speaker: str
    attack: str
    utt: str


def load_partition(root: Path, part: str):
    proto = root / "ASVspoof2019_LA_cm_protocols" / PROTOCOLS[part]
    flac = root / f"ASVspoof2019_LA_{part}" / "flac"
    if not proto.exists():
        return []
    items = []
    missing = 0
    for line in proto.read_text(encoding="utf-8").splitlines():
        f = line.split()
        if len(f) < 5:
            continue
        spk, utt, attack, key = f[0], f[1], f[3], f[4]
        p = flac / f"{utt}.flac"
        if not p.exists():
            missing += 1
            continue
        items.append(Item(str(p), 1 if key == "spoof" else 0, spk, attack, utt))
    if missing:
        print(f"  {part}: {missing} protocol rows have no flac on disk, skipped")
    return items


PARTS = {}
if LA_ROOT is not None:
    for part in PROTOCOLS:
        items = load_partition(LA_ROOT, part)
        PARTS[part] = items
        if items:
            labs = Counter(i.label for i in items)
            print(f"{part:<6} {len(items):>6} usable  bonafide {labs[0]:>5}  "
                  f"spoof {labs[1]:>6}  speakers {len({i.speaker for i in items}):>3}  "
                  f"attacks {sorted({i.attack for i in items if i.attack != '-'})}")
        else:
            print(f"{part:<6} nothing usable")

# Speaker disjointness: check it, do not assume it.
if PARTS:
    spk = {k: {i.speaker for i in v} for k, v in PARTS.items() if v}
    names = sorted(spk)
    print()
    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            x, y = names[a], names[b]
            n = len(spk[x] & spk[y])
            print(f"speaker overlap {x}/{y}: {n}  "
                  f"{'ok' if n == 0 else 'LEAK, results are not trustworthy'}")

## Dataset and loader

One item is one 4 second crop at 8 kHz, which is 32000 samples and matches
`rawnet.SEGMENT_S`. Four seconds holds several syllables and keeps a batch of
64 comfortably inside a T4.

Training takes a random crop each epoch. That is the cheapest useful
augmentation here: it stops the network from learning where in the file each
utterance starts, and it multiplies the effective corpus by the number of
distinct crops. Evaluation takes a deterministic crop from the beginning so
the numbers are reproducible.

Resampling goes through `swarkavach.audioio.resample`, the same windowed-sinc
resampler the laptop uses, so a checkpoint trained here behaves identically
when the laptop scores a call with it. A different resampler on each side is a
train-test mismatch that is very easy to miss.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from swarkavach.audioio import pad_or_trim, resample, rms_normalize
from swarkavach.config import SETTINGS, TARGET_SR, WIDEBAND_SR

SEGMENT_S = 4.0
N_SAMPLES = int(TARGET_SR * SEGMENT_S)
SEED = SETTINGS.pipeline.seed

try:
    import soundfile as sf
except Exception as exc:
    raise RuntimeError("soundfile is required to read flac") from exc


def read_any(path: str) -> np.ndarray:
    """Read flac or wav, mono, resampled to 8 kHz, RMS normalised."""
    x, sr = sf.read(path, dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if int(sr) != TARGET_SR:
        x = resample(np.asarray(x, dtype=np.float32), int(sr), TARGET_SR)
    return rms_normalize(np.asarray(x, dtype=np.float32))


class SpoofSet(Dataset):
    """Fixed length crops from a list of Items."""

    def __init__(self, items, train: bool, seed: int = SEED):
        self.items = list(items)
        self.train = bool(train)
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        try:
            x = read_any(it.path)
        except Exception:
            x = np.zeros(N_SAMPLES, dtype=np.float32)
        if x.size <= N_SAMPLES:
            x = pad_or_trim(x, N_SAMPLES)
        else:
            start = int(self.rng.integers(0, x.size - N_SAMPLES)) if self.train else 0
            x = x[start:start + N_SAMPLES]
        return torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), int(it.label)


def subsample(items, cap: int, seed: int = SEED):
    """Cap the size of a partition while keeping the class ratio."""
    if len(items) <= cap:
        return list(items)
    rng = np.random.default_rng(seed)
    by = {}
    for it in items:
        by.setdefault(it.label, []).append(it)
    out = []
    for lab, group in by.items():
        take = max(1, int(round(cap * len(group) / len(items))))
        idx = rng.choice(len(group), min(take, len(group)), replace=False)
        out.extend(group[j] for j in idx)
    rng.shuffle(out)
    return out


TRAIN_ITEMS = subsample(PARTS.get("train", []), MAX_TRAIN)
DEV_ITEMS = subsample(PARTS.get("dev", []), min(MAX_EVAL, 12000))
EVAL_ITEMS = subsample(PARTS.get("eval", []), MAX_EVAL)

print(f"train {len(TRAIN_ITEMS)}  dev {len(DEV_ITEMS)}  eval {len(EVAL_ITEMS)}")
if TRAIN_ITEMS:
    t0 = __import__("time").time()
    _ = SpoofSet(TRAIN_ITEMS[:32], train=True)[0]
    print(f"one item in {(__import__('time').time() - t0) * 1000:.0f} ms")

## Training

`RawNetLite` is imported from the repository rather than redefined here, so
the network that trains on Colab is bit for bit the network the laptop loads.
About 90 thousand parameters: a SincConv front end whose only free parameters
are the band edges, four pre-activation residual blocks, a GRU and a two-way
head.

The class weighting matters. LA train is roughly nine spoofed utterances to
one bona fide, so an unweighted loss learns "say spoof" and reaches 90 percent
accuracy while being worthless. The weights are the inverse class frequencies,
normalised to mean 1.

Checkpointing after every epoch, not at the end. A Colab session disconnects
without warning and a 40 minute run with nothing on disk is 40 minutes gone.
With Drive mounted the checkpoint also goes there, which is the only copy that
survives the session.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

from swarkavach.antispoof.rawnet import RawNetLite, load_rawnet, rawnet_path

CKPT = Path(rawnet_path("antispoof_rawnet.pt"))
CKPT.parent.mkdir(parents=True, exist_ok=True)

DRIVE_CKPT = None
_drive = Path("/content/drive/MyDrive/swarkavach")
if _drive.exists():
    DRIVE_CKPT = _drive / "antispoof_rawnet.pt"
    print(f"epoch checkpoints also go to {DRIVE_CKPT}")

HISTORY = {"train_loss": [], "dev_eer": [], "epoch_seconds": []}
TRAINED_ON = "none"

if not TRAIN_ITEMS:
    print("No ASVspoof training items. Skipping the RawNet training cell.")
    print("Notebook 04 will fall back to the locally trained anti-spoof branch.")
    net = None
else:
    torch.manual_seed(SEED)
    np.random.seed(SEED % (2 ** 31))

    net = RawNetLite(sr=TARGET_SR, segment_s=SEGMENT_S, seed=SEED).to(DEVICE)
    print(f"RawNetLite: {net.n_parameters():,} parameters")

    counts = np.bincount([i.label for i in TRAIN_ITEMS], minlength=2).astype(np.float64)
    weights = (counts.sum() / np.maximum(counts, 1.0)) / 2.0
    print(f"class counts bonafide={int(counts[0])} spoof={int(counts[1])}, "
          f"loss weights {np.round(weights, 3).tolist()}")

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    train_loader = DataLoader(SpoofSet(TRAIN_ITEMS, train=True), batch_size=BATCH,
                              shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
                              pin_memory=HAS_GPU)


    @torch.no_grad()
    def score_items(model, items, batch=None):
        """P(spoof) per utterance, deterministic crop, as a numpy array."""
        if not items:
            return np.zeros(0), np.zeros(0, dtype=int)
        model.eval()
        loader = DataLoader(SpoofSet(items, train=False), batch_size=batch or BATCH,
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=HAS_GPU)
        out, ys = [], []
        for xb, yb in loader:
            logits = model(xb.to(DEVICE, non_blocking=True))
            out.append(torch.softmax(logits, dim=1)[:, 1].float().cpu().numpy())
            ys.append(yb.numpy())
        return np.concatenate(out), np.concatenate(ys)


    from swarkavach.evaluate import eer as compute_eer

    best_dev = 1.0
    for epoch in range(EPOCHS):
        t0 = time.time()
        net.train()
        losses = []
        for step, (xb, yb) in enumerate(train_loader):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = criterion(net(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
            losses.append(float(loss.item()))
            if step % 100 == 0:
                print(f"  epoch {epoch + 1:2d} step {step:4d}  loss {np.mean(losses[-100:]):.4f}",
                      flush=True)
        sched.step()

        train_loss = float(np.mean(losses)) if losses else float("nan")
        dev_eer = float("nan")
        if DEV_ITEMS:
            s, y = score_items(net, DEV_ITEMS[:6000])
            dev_eer, _ = compute_eer(s, y)
        HISTORY["train_loss"].append(train_loss)
        HISTORY["dev_eer"].append(float(dev_eer))
        HISTORY["epoch_seconds"].append(round(time.time() - t0, 1))
        print(f"epoch {epoch + 1:2d}  train_loss {train_loss:.4f}  "
              f"dev EER {dev_eer * 100:.2f}%  {time.time() - t0:.0f}s")

        # Checkpoint every epoch, and keep the best dev EER separately.
        meta = {"epoch": epoch + 1, "history": HISTORY, "reduced": REDUCED,
                "n_train": len(TRAIN_ITEMS), "gpu": GPU}
        net.save(CKPT, meta=meta)
        if DRIVE_CKPT is not None:
            net.save(DRIVE_CKPT, meta=meta)
        if np.isfinite(dev_eer) and dev_eer < best_dev:
            best_dev = dev_eer
            net.save(CKPT.with_name("antispoof_rawnet_best.pt"), meta=meta)

    TRAINED_ON = "asvspoof2019_la"
    print(f"\ndone. best dev EER {best_dev * 100:.2f}%  checkpoint {CKPT} "
          f"({CKPT.stat().st_size / 1e6:.2f} MB)")

In [ ]:
import matplotlib.pyplot as plt

if HISTORY["train_loss"]:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ep = range(1, len(HISTORY["train_loss"]) + 1)
    ax[0].plot(ep, HISTORY["train_loss"], marker="o")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss")
    ax[0].set_title("training loss")
    ax[1].plot(ep, [e * 100 for e in HISTORY["dev_eer"]], marker="o", color="tab:orange")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("dev EER %")
    ax[1].set_title("dev EER, official dev partition")
    for a in ax:
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("data/results/rawnet_training.png", dpi=130)
    plt.show()
else:
    print("nothing trained, no curve")

## The classical baselines

LFCC-GMM is the ASVspoof 2019 and 2021 organisers' own baseline, and GFCC-GMM
is the same machinery on an ERB-spaced gammatone filterbank. Both come from
this repository (`antispoof.features` and `antispoof.gmm`), so the front ends
are identical to the ones the laptop runs.

They are here for two reasons. First, a neural model that cannot beat a
two-mixture-model baseline is not worth the checkpoint. Second, Plan B section
1 says that if the neural branch will not train reliably, these become the
anti-spoof branch on their own, and the comparison table is the evidence for
whether that is an acceptable outcome.

Frame extraction is the slow part on CPU, so the number of utterances per
class is capped by `GMM_UTTS`.

In [ ]:
import time

import numpy as np

from swarkavach.antispoof.features import frame_features
from swarkavach.antispoof.gmm import GMMScorer

GMM_MODELS = {}


def frames_for(items, feature_set, cap, tag=""):
    """Stack frame features for a list of items, class balanced."""
    bona = [i for i in items if i.label == 0][:cap]
    spoof = [i for i in items if i.label == 1][:cap]
    out = {}
    for name, group in (("bona", bona), ("spoof", spoof)):
        mats = []
        t0 = time.time()
        for k, it in enumerate(group):
            try:
                mats.append(frame_features(read_any(it.path), TARGET_SR, feature_set))
            except Exception:
                continue
            if k and k % 500 == 0:
                print(f"    {tag}{feature_set} {name} {k}/{len(group)} "
                      f"({time.time() - t0:.0f}s)", flush=True)
        out[name] = np.vstack(mats) if mats else np.zeros((0, 1))
    return out["bona"], out["spoof"]


if TRAIN_ITEMS:
    for fs in ("lfcc", "gfcc"):
        print(f"\nfitting {fs.upper()}-GMM on up to {GMM_UTTS} utterances per class")
        t0 = time.time()
        bona, spoof = frames_for(TRAIN_ITEMS, fs, GMM_UTTS, tag="train ")
        print(f"  frames: bonafide {bona.shape}, spoof {spoof.shape}")
        try:
            g = GMMScorer(feature_set=fs).fit(bona, spoof)
            GMM_MODELS[fs] = g
            print(f"  {g}  fitted in {time.time() - t0:.0f}s")
        except Exception as exc:
            print(f"  {fs} GMM failed: {type(exc).__name__}: {exc}")
else:
    print("no training items, skipping the GMM baselines")

## Evaluation

Two numbers per model per condition.

**EER** is the operating point where the false alarm rate equals the miss
rate. It is threshold free, which is why the anti-spoofing literature reports
it, and it is the number to compare across systems.

**min t-DCF**, as computed by `swarkavach.evaluate.min_tdcf`, is a
cost-weighted operating point with the speaker verification stage held fixed
(`p_target=0.05`, `c_miss=1.0`, `c_fa=10.0`). It is not the full tandem
detection cost from the ASVspoof evaluation plan, because there is no ASV
system in this loop. The function's docstring says so and the report should
too, otherwise the number reads as more comparable to published t-DCF figures
than it is.

In [ ]:
import time

import numpy as np

from swarkavach.antispoof.features import frame_features
from swarkavach.evaluate import det_curve, eer, min_tdcf

SCORES = {}          # (model, condition) -> (scores, labels, attacks)


def det_points(scores, labels, max_points: int = 90):
    fpr, fnr, _ = det_curve(scores, labels)
    step = max(1, len(fpr) // max_points)
    return [[round(float(a), 5), round(float(b), 5)]
            for a, b in zip(fpr[::step], fnr[::step])]


def gmm_scores(model, items, label_tag=""):
    out = []
    t0 = time.time()
    for k, it in enumerate(items):
        try:
            out.append(float(model.llr(frame_features(read_any(it.path), TARGET_SR,
                                                      model.feature_set))))
        except Exception:
            out.append(0.0)
        if k and k % 2000 == 0:
            print(f"    {label_tag} {k}/{len(items)} ({time.time() - t0:.0f}s)", flush=True)
    return np.asarray(out, dtype=float)


# ---- ASVspoof eval partition ------------------------------------------------
EVAL_USED = EVAL_ITEMS or DEV_ITEMS
EVAL_NAME = "asvspoof_eval" if EVAL_ITEMS else "asvspoof_dev"
if EVAL_USED:
    y = np.asarray([i.label for i in EVAL_USED], dtype=int)
    attacks = [i.attack for i in EVAL_USED]

    if net is not None:
        s, y_chk = score_items(net, EVAL_USED)
        assert (y_chk == y).all(), "loader reordered the eval set"
        SCORES[("rawnet", EVAL_NAME)] = (s, y, attacks)
        e, _ = eer(s, y)
        print(f"RawNetLite  {EVAL_NAME}  EER {e * 100:.2f}%  "
              f"min t-DCF {min_tdcf(s, y):.4f}")

    for fs, g in GMM_MODELS.items():
        s = gmm_scores(g, EVAL_USED, label_tag=f"{fs} eval")
        SCORES[(f"{fs}_gmm", EVAL_NAME)] = (s, y, attacks)
        e, _ = eer(s, y)
        print(f"{fs.upper()}-GMM   {EVAL_NAME}  EER {e * 100:.2f}%  "
              f"min t-DCF {min_tdcf(s, y):.4f}")
else:
    print("no ASVspoof evaluation items")

## In-the-Wild, and why the gap is the interesting part

In-the-Wild is 58 public figures, genuine and deepfaked, collected from public
video rather than generated for a benchmark. Published results put ASVspoof
trained detectors around 30 to 45 percent EER on it, against a few percent on
ASVspoof eval.

Expect the same here, and do not read it as a broken model. ASVspoof LA is
studio-clean, single channel, one recording setup, thirteen known attack
algorithms. In-the-Wild is compressed social media audio, background music,
unknown attacks, real acoustics. A detector trained on the first has learned
"what a vocoder does to clean studio speech", and the second asks a different
question.

That gap is the argument for this whole project. A single-branch audio
detector is not a fraud detector in the wild, which is precisely why the text
branch and the cross-modal features exist. Report the gap, do not hide it.

In [ ]:
import csv
from pathlib import Path

import numpy as np

ITW_ITEMS = []
if ITW_ROOT is not None:
    metas = sorted(Path(ITW_ROOT).glob("**/*.csv"))
    wavs = {p.name: p for p in Path(ITW_ROOT).glob("**/*.wav")}
    if metas and wavs:
        rows = list(csv.DictReader(metas[0].open(encoding="utf-8")))
        lab_col = next(c for c in rows[0] if "label" in c.lower())
        file_col = next(c for c in rows[0] if "file" in c.lower())
        spk_col = next((c for c in rows[0] if "speaker" in c.lower()), None)
        for r in rows:
            p = wavs.get(r[file_col])
            if p is None:
                continue
            lab = r[lab_col].strip().lower()
            ITW_ITEMS.append(Item(str(p), 1 if lab.startswith("spoof") else 0,
                                  r.get(spk_col, "unknown") if spk_col else "unknown",
                                  "in_the_wild", p.stem))
        ITW_ITEMS = subsample(ITW_ITEMS, MAX_ITW)
        labs = np.bincount([i.label for i in ITW_ITEMS], minlength=2)
        print(f"In-the-Wild: {len(ITW_ITEMS)} clips, bonafide {labs[0]}, spoof {labs[1]}, "
              f"{len({i.speaker for i in ITW_ITEMS})} speakers")
    else:
        print("In-the-Wild files found but no usable metadata csv")
else:
    print("In-the-Wild not on disk, skipping the generalisation test. "
          "The export marks the condition as not run rather than as zero.")

if ITW_ITEMS:
    y_itw = np.asarray([i.label for i in ITW_ITEMS], dtype=int)
    if net is not None:
        s, y_chk = score_items(net, ITW_ITEMS)
        SCORES[("rawnet", "in_the_wild")] = (s, y_itw, ["in_the_wild"] * len(y_itw))
        e, _ = eer(s, y_itw)
        print(f"RawNetLite  in_the_wild  EER {e * 100:.2f}%  min t-DCF {min_tdcf(s, y_itw):.4f}")
    for fs, g in GMM_MODELS.items():
        s = gmm_scores(g, ITW_ITEMS, label_tag=f"{fs} itw")
        SCORES[(f"{fs}_gmm", "in_the_wild")] = (s, y_itw, ["in_the_wild"] * len(y_itw))
        e, _ = eer(s, y_itw)
        print(f"{fs.upper()}-GMM   in_the_wild  EER {e * 100:.2f}%  "
              f"min t-DCF {min_tdcf(s, y_itw):.4f}")

In [ ]:
import numpy as np
import pandas as pd

from swarkavach.evaluate import eer, min_tdcf

rows = []
for (model, cond), (s, y, _) in sorted(SCORES.items()):
    if len(set(y.tolist())) < 2:
        continue
    e, thr = eer(s, y)
    rows.append({"model": model, "condition": cond, "n": len(y),
                 "EER %": round(e * 100, 2),
                 "min t-DCF": round(min_tdcf(s, y), 4),
                 "threshold": round(float(thr), 4)})

TABLE = pd.DataFrame(rows)
if not TABLE.empty:
    print(TABLE.to_string(index=False))
    wide = TABLE.pivot(index="model", columns="condition", values="EER %")
    print("\nEER % by model and condition:")
    print(wide.to_string())
    if EVAL_NAME in wide.columns and "in_the_wild" in wide.columns:
        gap = wide["in_the_wild"] - wide[EVAL_NAME]
        print("\ngeneralisation gap (in_the_wild minus benchmark), percentage points:")
        print(gap.round(2).to_string())
else:
    print("nothing scored")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from swarkavach.evaluate import det_curve, eer

DET_STORE = []
fig, ax = plt.subplots(figsize=(5.6, 5.2))
for (model, cond), (s, y, _) in sorted(SCORES.items()):
    if len(set(y.tolist())) < 2:
        continue
    fpr, fnr, _ = det_curve(s, y)
    e, _ = eer(s, y)
    name = f"{model} / {cond}"
    ax.plot(fpr * 100, fnr * 100, label=f"{name}  EER {e * 100:.1f}%")
    DET_STORE.append({"name": name, "points": det_points(s, y), "eer": round(float(e), 4)})

ax.plot([0, 60], [0, 60], "k--", lw=0.8, alpha=0.5)
ax.set_xlim(0, 60); ax.set_ylim(0, 60)
ax.set_xlabel("false alarm rate %"); ax.set_ylabel("miss rate %")
ax.set_title("DET curves")
ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig("data/results/antispoof_det.png", dpi=130)
plt.show()
print(f"{len(DET_STORE)} DET series stored for the console")

## Branch scores for notebook 04

The trained checkpoint is the real-data audio branch. Notebook 04 needs one
P(synthetic) per generated corpus call so the fusion vector's `voice` group
carries a real-data number instead of a number learned from the same
synthesiser that produced the audio.

There is a caveat worth writing into the report: the generated corpus audio
comes from the repository's own source-filter synthesiser, not from a neural
vocoder, so a detector trained on ASVspoof attacks is being asked about a
signal unlike anything in its training set. Whatever it says here is itself a
small generalisation experiment.

In [ ]:
import json
from pathlib import Path

import numpy as np

BRANCH = {}
if net is not None:
    try:
        from swarkavach.corpus.generator import load_corpus

        calls = load_corpus()
    except Exception as exc:
        calls = []
        print(f"corpus not loadable ({type(exc).__name__}: {exc}), skipping branch scores")

    scored = 0
    for c in calls:
        if not c.audio_path or not Path(c.audio_path).exists():
            continue
        try:
            x = read_any(c.audio_path)
            with torch.no_grad():
                segs = []
                for start in range(0, max(len(x) - N_SAMPLES, 0) + 1, N_SAMPLES):
                    segs.append(pad_or_trim(x[start:start + N_SAMPLES], N_SAMPLES))
                if not segs:
                    segs = [pad_or_trim(x, N_SAMPLES)]
                xb = torch.from_numpy(np.stack(segs).astype(np.float32)).to(DEVICE)
                p = torch.softmax(net(xb), dim=1)[:, 1].float().cpu().numpy()
            BRANCH[c.call_id] = {
                "as_score": round(float(np.mean(p)), 5),
                "as_max_segment": round(float(np.max(p)), 5),
                "n_segments": int(len(segs)),
                "label_voice": c.label_voice,
            }
            scored += 1
        except Exception:
            continue
    print(f"scored {scored} corpus calls with the ASVspoof-trained RawNet")

    if BRANCH:
        syn = [v["as_score"] for v in BRANCH.values() if v["label_voice"] == "synthetic"]
        hum = [v["as_score"] for v in BRANCH.values() if v["label_voice"] == "human"]
        if syn and hum:
            print(f"  mean P(synthetic): synthetic cells {np.mean(syn):.3f}, "
                  f"human cells {np.mean(hum):.3f}")
else:
    print("no trained network, no branch scores")

## Export

`antispoof_results.json` follows the shape `evaluate._eval_antispoof` writes,
so the console's Results lab renders it without any frontend change:
`eer[feature_set][model][condition]`, the same for `min_tdcf`, and `det` as a
list of named point series. Here the feature sets are `raw` (the waveform,
which is what RawNetLite consumes), `lfcc` and `gfcc`, and the conditions are
the ASVspoof partition and In-the-Wild.

In [ ]:
# ---------------------------------------------------------------------------
# Export helpers. Each notebook carries its own copy so it can run on its own.
#
# Everything written here lands in data/results/ with "source": "colab", a UTC
# timestamp and the GPU name, which is what the local console's provenance
# table reads to tell your laptop numbers apart from your Colab numbers.
# ---------------------------------------------------------------------------
import json
import platform
import zipfile
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK = "02_antispoof_train"
RESULTS = Path("data/results")
RESULTS.mkdir(parents=True, exist_ok=True)


def gpu_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return "none (CPU runtime)"


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


def colab_stamp(notes: str = "") -> dict:
    try:
        import torch

        tv = str(torch.__version__)
    except Exception:
        tv = "not installed"
    return {
        "source": "colab",
        "generated": utc_now(),
        "gpu": gpu_name(),
        "torch": tv,
        "python": platform.python_version(),
        "notebook": NOTEBOOK,
        "notes": notes,
    }


WRITTEN = []


def write_result(name: str, obj: dict, notes: str = "") -> Path:
    """Write one result file, stamped, and remember it for the zip."""
    doc = dict(obj)
    doc.update(colab_stamp(notes))
    path = RESULTS / name
    path.write_text(json.dumps(doc, indent=2, default=float), encoding="utf-8")
    if name not in WRITTEN:
        WRITTEN.append(name)
    print(f"wrote {path}  ({path.stat().st_size / 1024:.1f} kB)")
    return path


def update_provenance(entries) -> Path:
    """Merge these Colab entries into data/results/provenance.json.

    Entries for other artifacts are kept, so a local `swarkavach evaluate` run
    and a Colab run can sit in the same table. Unzip the Colab results AFTER
    the local evaluate, otherwise the local file overwrites this one.
    """
    path = RESULTS / "provenance.json"
    doc = {"entries": []}
    if path.exists():
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    new_names = {e["artifact"] for e in entries}
    kept = [e for e in doc.get("entries", []) if e.get("artifact") not in new_names]
    doc["entries"] = kept + list(entries)
    doc["generated"] = utc_now()
    doc["host"] = platform.platform()
    doc["gpu"] = gpu_name()
    path.write_text(json.dumps(doc, indent=2), encoding="utf-8")
    if "provenance.json" not in WRITTEN:
        WRITTEN.append("provenance.json")
    print(f"provenance now lists {len(doc['entries'])} artifacts")
    return path


def export_zip(extra_paths=(), zip_name: str = None) -> Path:
    """Zip the result files plus anything extra, then hand it to the browser."""
    out = Path(zip_name or f"swarkavach_{NOTEBOOK}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in WRITTEN:
            f = RESULTS / name
            if f.exists():
                z.write(f, arcname=f"data/results/{f.name}")
        for extra in extra_paths:
            p = Path(extra)
            if p.exists():
                z.write(p, arcname=p.as_posix())
            else:
                print(f"  skipped missing {p}")

    names = zipfile.ZipFile(out).namelist()
    print(f"\n{out.name}  {out.stat().st_size / 1024:.1f} kB")
    for n in names:
        print("   ", n)

    try:
        from google.colab import files

        files.download(str(out))
        print("\nDownload started. Unzip it over your local project root.")
    except Exception as exc:
        print(f"\nnot running on Colab ({type(exc).__name__}), zip is at {out.resolve()}")
    return out

In [ ]:
import numpy as np

from swarkavach.evaluate import eer, min_tdcf

FEATURE_SETS = ["raw", "lfcc", "gfcc"]
MODEL_NAMES = ["rawnet", "gmm"]
CONDITIONS = [c for c in [EVAL_NAME if EVAL_USED else None,
                          "in_the_wild" if ITW_ITEMS else None] if c]

# (feature set, model) -> the key used in SCORES
LAYOUT = {("raw", "rawnet"): "rawnet",
          ("lfcc", "gmm"): "lfcc_gmm",
          ("gfcc", "gmm"): "gfcc_gmm"}

eer_tbl, tdcf_tbl, by_attack = {}, {}, {}
for fs in FEATURE_SETS:
    eer_tbl[fs], tdcf_tbl[fs] = {}, {}
    for mdl in MODEL_NAMES:
        key = LAYOUT.get((fs, mdl))
        if key is None:
            continue
        eer_tbl[fs][mdl], tdcf_tbl[fs][mdl] = {}, {}
        for cond in CONDITIONS:
            entry = SCORES.get((key, cond))
            if entry is None or len(set(entry[1].tolist())) < 2:
                eer_tbl[fs][mdl][cond] = None
                tdcf_tbl[fs][mdl][cond] = None
                continue
            s, y, attacks = entry
            e, _ = eer(s, y)
            eer_tbl[fs][mdl][cond] = round(float(e), 4)
            tdcf_tbl[fs][mdl][cond] = round(float(min_tdcf(s, y)), 4)

            # Per attack, bona fide against that attack alone. An aggregate EER
            # hides a model that is near perfect on six attacks and blind to one.
            if cond.startswith("asvspoof"):
                bona = np.flatnonzero(y == 0)
                for a in sorted({x for x in attacks if x not in ("-", "in_the_wild")}):
                    idx = np.array([i for i, x in enumerate(attacks) if x == a])
                    if idx.size < 5:
                        continue
                    sel = np.concatenate([bona, idx])
                    ea, _ = eer(s[sel], y[sel])
                    by_attack.setdefault(f"{fs}/{mdl}", {})[a] = round(float(ea), 4)

results = {
    "feature_sets": FEATURE_SETS,
    "models": MODEL_NAMES,
    "conditions": CONDITIONS,
    "default_condition": CONDITIONS[0] if CONDITIONS else "asvspoof_eval",
    "eer": eer_tbl,
    "min_tdcf": tdcf_tbl,
    "det": DET_STORE,
    "by_attack": by_attack,
    "n_train": len(TRAIN_ITEMS),
    "n_dev": len(DEV_ITEMS),
    "n_test": len(EVAL_USED) if EVAL_USED else 0,
    "n_in_the_wild": len(ITW_ITEMS),
    "trained_on": TRAINED_ON,
    "reduced": REDUCED,
    "epochs": EPOCHS,
    "history": HISTORY,
    "sample_rate": TARGET_SR,
    "segment_s": SEGMENT_S,
    "note": ("min_tdcf here is a cost-weighted operating point with the ASV stage "
             "fixed (p_target 0.05, c_miss 1, c_fa 10), not the full tandem t-DCF. "
             "Audio is resampled to 8 kHz to match the project's telephone-band "
             "design, which costs EER against 16 kHz numbers in the literature."),
}

write_result("antispoof_results.json", results,
             notes=f"RawNetLite + LFCC/GFCC-GMM on {TRAINED_ON}, {EPOCHS} epochs")

if BRANCH:
    write_result("antispoof_branch_scores.json",
                 {"scores": BRANCH, "model": "rawnet_asvspoof",
                  "checkpoint": str(CKPT), "n": len(BRANCH)},
                 notes="P(synthetic) per generated corpus call, ASVspoof-trained RawNet")

entries = [{
    "artifact": "antispoof_results.json", "source": "colab", "generated": utc_now(),
    "notes": (f"RawNetLite trained on {TRAINED_ON} ({len(TRAIN_ITEMS)} utterances, "
              f"{EPOCHS} epochs) on {gpu_name()}"
              + (", reduced CPU configuration" if REDUCED else "")),
}]
if BRANCH:
    entries.append({
        "artifact": "antispoof_branch_scores.json", "source": "colab",
        "generated": utc_now(),
        "notes": f"{len(BRANCH)} generated corpus calls scored by the real-data model",
    })
update_provenance(entries)

export_zip(extra_paths=[
    CKPT,
    Path("data/results/antispoof_det.png"),
    Path("data/results/rawnet_training.png"),
])

## Next

`03_asr_ner_intent.ipynb` for Whisper, the gold versus ASR entity table and
the MuRIL intent fine-tune. Notebook 04 needs both this notebook's
`antispoof_branch_scores.json` and notebook 03's `intent_branch_scores.json`.

To use the checkpoint on the laptop: unzip the download over the project root
so `data/models/antispoof_rawnet.pt` lands in place, then
`swarkavach info` will show the rawnet back end as available.